## Cleaning of the Huang Award Data

### Objective
This notebook cleans and filters the Huang Award dataset to prepare it for analysis of best paper trajectories. The workflow includes:
1. **Loading** the raw scraped data and inspecting its structure
2. **Cleaning** by removing missing values, duplicates, and standardizing text
3. **Merging** ICWSM (Best Paper) and JCDL (Vannevar Bush Best Paper) from `icwsm_jcdl_awards_raw.csv`
4. **Removing** NSDI (excluded from analysis)
5. **Filtering** to a pilot period (2000-2018) for focused analysis
6. **Saving** cleaned and filtered datasets for downstream analysis

In [3]:
import pandas as pd

# Load your scraped data
df = pd.read_csv("B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\raw\\huang_awards_complete_sel.csv")

print(f"Total awards: {len(df)}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"Unique conferences: {df['conference'].nunique()}")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nSample:")
print(df.head(10))


Total awards: 1507
Year range: 1996 - 2023
Unique conferences: 32

Missing values:
year           0
conference     0
paper_title    0
paper_url      0
authors        0
dtype: int64

Sample:
   year conference                                        paper_title  \
0  2023       AAAI  Misspecification in Inverse Reinforcement Lear...   
1  2023        ACL  Do Androids Laugh at Electric Sheep? Humor "Un...   
2  2023        ACL  What the DAAM: Interpreting Stable Diffusion U...   
3  2023        ACL  From Pretraining Data to Language Models to Do...   
4  2023        CHI  Breaking Out of the Ivory Tower: A Large-scale...   
5  2023        CHI  Changes in Research Ethics, Openness, and Tran...   
6  2023        CHI  ChartDetective: Easy and Accurate Interactive ...   
7  2023        CHI  CiteSee: Augmenting Citations in Scientific Pa...   
8  2023        CHI  Collaborating Across Realities: Analytical Len...   
9  2023        CHI  Contestable Camera Cars: A Speculative Design ...   

      

## Cleaning

In [4]:
# 1. Remove rows with missing critical fields
df_clean = df.dropna(subset=['year', 'conference', 'paper_title']).copy()

# 2. Remove duplicates (same paper title + year)
df_clean = df_clean.drop_duplicates(subset=['paper_title', 'year'], keep='first')

# 3. Clean paper titles (remove extra whitespace)
df_clean['paper_title'] = df_clean['paper_title'].str.strip()

## Merge ICWSM & JCDL, Remove NSDI

- **Add** ICWSM Best Paper awards and JCDL Vannevar Bush Best Paper awards from `icwsm_jcdl_awards_raw.csv`
- **Remove** NSDI (excluded from analysis scope)

In [ ]:
SHARED_COLS = ['year', 'conference', 'paper_title', 'paper_url', 'authors']

df_extra = pd.read_csv("B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\raw\\icwsm_jcdl_awards_raw.csv")

df_extra = df_extra[
    ((df_extra['conference'] == 'ICWSM') & (df_extra['award_type'] == 'Best Paper')) |
    ((df_extra['conference'] == 'JCDL')  & (df_extra['award_type'] == 'Vannevar Bush Best Paper'))
][SHARED_COLS].copy()

df_extra['paper_title'] = df_extra['paper_title'].str.strip()

# Merge and remove NSDI
df_clean = pd.concat([df_clean[SHARED_COLS], df_extra], ignore_index=True)
df_clean = df_clean[df_clean['conference'] != 'NSDI']
df_clean = df_clean.drop_duplicates(subset=['paper_title', 'year'], keep='first').reset_index(drop=True)

print(f"After merge + NSDI removal: {len(df_clean)} awards")
print(f"Unique conferences: {df_clean['conference'].nunique()}")
print(df_clean[df_clean['conference'].isin(['ICWSM', 'JCDL'])].groupby('conference')['year'].agg(['count', 'min', 'max']))

## Filtering

In [6]:
# ── Cell 3: Filtering ─────────────────────────────────────────────────────────
START_YEAR = 2000
END_YEAR = 2018

df_pilot = df_clean[
    (df_clean['year'].astype(int) >= START_YEAR) &
    (df_clean['year'].astype(int) <= END_YEAR)
].copy()

print("=" * 60)
print(f"After cleaning:  Total {len(df_clean)} awards")
print(f"Pilot {START_YEAR}-{END_YEAR}, ALL conferences: {len(df_pilot)} awards")
print(f"Unique conferences: {df_pilot['conference'].nunique()}")
print("\nBreakdown by conference:")
print(df_pilot.groupby('conference')['year'].agg(['count', 'min', 'max']))

# Save
df_clean.to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\cleaned\\huang_awards_cleaned.csv', index=False)
df_pilot.to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\cleaned\\huang_awards_pilot.csv', index=False)
print("\nSaved huang_awards_cleaned.csv and huang_awards_pilot.csv")


After cleaning:  Total 1506 awards
Pilot 2000-2018, ALL conferences: 912 awards
Unique conferences: 30

Breakdown by conference:
            count   min   max
conference                   
AAAI           26  2000  2018
ACL            26  2001  2018
CHI           192  2005  2018
CIKM           16  2004  2018
CVPR           20  2000  2018
FOCS           31  2002  2018
FSE            63  2002  2018
ICCV           12  2001  2017
ICML           19  2005  2018
ICSE           88  2003  2018
IJCAI          27  2001  2018
INFOCOM        24  2000  2018
KDD            20  2000  2018
MOBICOM        11  2008  2018
NeurIPS        15  2013  2018
OSDI           21  2000  2018
PLDI           32  2000  2018
PODS           20  2000  2018
S&P            13  2008  2018
SIGCOMM        15  2008  2018
SIGIR          19  2000  2018
SIGMETRICS     15  2004  2018
SIGMOD         19  2000  2018
SODA           15  2009  2018
SOSP           25  2001  2017
STOC           31  2003  2018
UIST           35  2000  2018
V